# 13G to 13D flips: passive holders turning active

Find holders who filed a passive 13G on a company before and now filed a 13D, the switch that signals an intent to influence.

Read it with charts at https://www.quantisnow.com/sec-api/guides/13g-to-13d-flips. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

A 13G is for passive holders. A holder who decides to push for change has to switch to a 13D within ten days and stop buying for ten more. So a 13D from someone who filed a 13G on the same company before is a holder changing its mind.

## Start from this month's 13Ds

### Which activist 13D stakes were filed in the last 30 days?

A 13D is due within five business days of crossing 5%. The list is short enough to read every week.

```shell
curl -X POST https://www.quantisnow.com/api/v1/ownership \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "formType:13D AND filedAt:[2026-10-25 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"filedAt": {"order": "desc"}}]
}'
```

Response (real, first 2 of 56):

```json
{
  "total": {
    "value": 56,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001418812-26-000003",
      "accessionNo": "0001418812-26-000003",
      "formType": "13D",
      "filedAt": "2026-11-23T18:36:03-05:00",
      "filer": {
        "cik": "0001418812",
        "name": "VA Partners I, LLC"
      },
      "subject": {
        "cik": "0001576789",
        "name": "Wix.com Ltd.",
        "ticker": "WIX"
      },
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1576789/000141881226000003/0001418812-26-000003-index.htm"
    },
    {
      "id": "0002141359-26-000002",
      "accessionNo": "0002141359-26-000002",
      "formType": "13D",
      "filedAt": "2026-11-23T16:11:08-05:00",
      "filer": {
        "cik": "0002141359",
        "name": "Wall Douglas Scott"
      },
      "subject": {
        "cik": "0001903595",
        "name": "Brag House Holdings Inc.",
        "ticker": "TBH"
      },
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1903595/000214135926000002/0002141359-26-000002-index.htm"
    }
  ]
}
```

## In Python

In [ ]:
import os, requests

URL = "https://www.quantisnow.com/api/v1/ownership"
H = {"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"}
def search(q, size=50):
    return requests.post(URL, headers=H, json={"query": q, "size": size}).json()

new_13d = search("formType:13D AND filedAt:[2026-10-24 TO 2026-11-23]")["data"]
for d in new_13d:
    filer, subject = d["filer"]["cik"], d["subject"]["cik"]
    prior = search(f"formType:13G AND filer.cik:{filer} AND subject.cik:{subject}", size=1)
    if prior["total"]["value"]:
        print("flip:", d["subject"]["ticker"], d["filer"]["name"], d["filedAt"][:10])

## From the command line

```shell
npx quantisnow query ownership 'formType:13D AND filedAt:[2026-10-24 TO 2026-11-23]' --sort filedAt:desc -n 50
```

Related: [13D plans and proposals](https://www.quantisnow.com/sec-api/guides/activist-13d-plans-or-proposals), [Activist 13D tracking](https://www.quantisnow.com/sec-api/guides/track-activist-13d-filings).